# 04 School predictors and district split

We prepare school characteristics for predicting AP participation in held-out districts. Notebook 05 fits the transformations and compares models.

### Imports

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42
TEST_SIZE = 0.2
DATA_DIR = Path("../data")
PROC_DIR = DATA_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

## Load the cleaned dataset

ID-like columns are read as strings so leading zeros survive.

In [6]:
df = pd.read_csv(DATA_DIR / "ap_clean_2021.csv",
                 dtype={"ncessch": str, "leaid": str, "state_location": str})
print("loaded:", df.shape)

loaded: (19416, 20)


## Choose predictors

We exclude both components of `ap_rate`, keep district IDs for splitting, and remove school identifiers and the duplicate state code.

In [7]:
y      = df["ap_rate"]
groups = df["leaid"]
X = df.drop(columns=[
    "ap_rate",       # target
    "enrl_9to12",
    "enrl_AP",       # leakage: numerator of the target
    "ncessch",       # identifier
    "school_name",   # identifier
    "leaid",         # grouping variable, handled separately
    "fips",          # numeric duplicate of state_location
])
print("feature matrix X:", X.shape)
print("features:", list(X.columns))

feature matrix X: (19416, 13)
features: ['state_location', 'latitude', 'longitude', 'urban_centric_locale', 'lowest_grade_offered', 'charter', 'magnet', 'title_i_status', 'enrollment', 'teachers_fte', 'student_teacher_ratio', 'free_or_reduced_price_lunch', 'frl_pct']


## Define feature types (and fix placeholder codes)

| type | features | encoding |
|---|---|---|
| **continuous (skewed)** | `enrollment`, `teachers_fte`, `student_teacher_ratio`, `free_or_reduced_price_lunch` | `log1p` → scale |
| **continuous (other)** | `latitude`, `longitude`, `frl_pct` | scale |
| **categorical** | `state_location`, `title_i_status`, `charter`, `magnet`, `urban_centric_locale` | one-hot (NaN = `'missing'`) |
| **ordinal** | `lowest_grade_offered` | ordinal (NaN = own lowest category) |

Two placeholder/implausible values are recoded to `NaN` (no rows dropped). These are deterministic
fixes with fixed thresholds — no statistics are learned — so they are leakage-safe before the split.
Categorical columns are stringified so their `NaN` becomes an explicit `'missing'` category.

In [8]:
cont_skewed = ["enrollment", "teachers_fte",
               "student_teacher_ratio", "free_or_reduced_price_lunch"]
cont_other  = ["latitude", "longitude", "frl_pct"]
cat_ftrs    = ["state_location", "title_i_status", "charter", "magnet",
               "urban_centric_locale"]
ord_ftrs    = ["lowest_grade_offered"]
continuous_ftrs = cont_skewed + cont_other
assert set(cont_skewed + cont_other + cat_ftrs + ord_ftrs) == set(X.columns)

X = X.copy()
# lowest_grade_offered == -1 is a placeholder (not a real grade) -> NaN
n_lgo = (X["lowest_grade_offered"] == -1).sum()
X.loc[X["lowest_grade_offered"] == -1, "lowest_grade_offered"] = np.nan
# implausible student-teacher ratios -> NaN
STR_CAP = 100
n_str = (X["student_teacher_ratio"] > STR_CAP).sum()
X.loc[X["student_teacher_ratio"] > STR_CAP, "student_teacher_ratio"] = np.nan
# categorical NaN -> explicit 'missing' category
X[cat_ftrs] = X[cat_ftrs].astype("string").fillna("missing")

print(f"recoded lowest_grade_offered == -1 : {n_lgo}")
print(f"recoded student_teacher_ratio > {STR_CAP}: {n_str}")
print("feature types:  %d skewed-cont, %d other-cont, %d categorical, %d ordinal"
      % (len(cont_skewed), len(cont_other), len(cat_ftrs), len(ord_ftrs)))

recoded lowest_grade_offered == -1 : 864
recoded student_teacher_ratio > 100: 27
feature types:  4 skewed-cont, 3 other-cont, 5 categorical, 1 ordinal


## Splitting strategy 

Why group-based? Schools in the same district (`leaid`) share policy and funding, so they are not
independent. We hold out whole districts with `GroupShuffleSplit` (`test_size=0.2`) so no district
appears in both the development and test sets  otherwise the test score would be optimistic.

We split first so that every subsequent step including the missing-value analysis below and all
fitted transformers touches the development set only, never the held-out test set.

In [9]:
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
other_idx, test_idx = next(gss.split(X, y, groups=groups))

X_other, X_test = X.iloc[other_idx].copy(), X.iloc[test_idx].copy()
y_other, y_test = y.iloc[other_idx].copy(), y.iloc[test_idx].copy()
groups_other, groups_test = groups.iloc[other_idx].copy(), groups.iloc[test_idx].copy()

assert not set(groups_other) & set(groups_test), "district overlap between dev and test!"
print(f"development (X_other): {X_other.shape} | test: {X_test.shape}")
print("district overlap:", len(set(groups_other) & set(groups_test)))

development (X_other): (15617, 13) | test: (3799, 13)
district overlap: 0


## Analyze missing values

Before choosing a strategy we look at how much is missing, which patterns occur, and whether
missingness is associated with the target or other features. if it is, missingness itself carries
information (which is an argument for letting XGBoost use the NaNs natively rather than imputing them away).

In [10]:
missing_summary = pd.DataFrame({
    "missing_count": X_other[continuous_ftrs].isna().sum(),
    "missing_pct":   X_other[continuous_ftrs].isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)
missing_summary.round(2)

,missing_count,missing_pct
frl_pct,2757,17.65
free_or_reduced_price_lunch,2751,17.62
student_teacher_ratio,453,2.90
teachers_fte,430,2.75
enrollment,0,0.00
latitude,0,0.00
longitude,0,0.00


In [ ]:
# which combinations of missing continuous features occur?
missing_patterns = X_other[continuous_ftrs].isna().value_counts()
print("Number of distinct missing-value patterns:", len(missing_patterns))
print(missing_patterns.head(10))

Number of distinct missing-value patterns: 8
enrollment  teachers_fte  student_teacher_ratio  free_or_reduced_price_lunch  latitude  longitude  frl_pct
False       False         False                  False                        False     False      False      12590
                                                 True                         False     False      True        2569
            True          True                   False                        False     False      False        261
                                                 True                         False     False      True         168
            False         True                   True                         False     False      True          14
                                                 False                        False     False      False          9
                          False                  False                        False     False      True           5
            True          True      

In [13]:
# Is missingness associated with the target / school size? Compare rows where a feature is
# missing vs present (development set only).
def missingness_vs_target(col):
    m = X_other[col].isna()
    return pd.Series({
        "n_missing":        int(m.sum()),
        "mean_ap_missing":  y_other[m].mean(),
        "mean_ap_present":  y_other[~m].mean(),
        "zeroshare_missing": (y_other[m] == 0).mean(),
        "zeroshare_present": (y_other[~m] == 0).mean(),
        "median_enrl_missing": X_other.loc[m, "enrollment"].median(),
        "median_enrl_present": X_other.loc[~m, "enrollment"].median(),
    })

assoc = pd.DataFrame({c: missingness_vs_target(c)
                      for c in ["frl_pct", "teachers_fte", "student_teacher_ratio"]}).T
print(assoc.round(3))

                       n_missing  mean_ap_missing  mean_ap_present  \
frl_pct                   2757.0            0.106            0.136   
teachers_fte               430.0            0.095            0.132   
student_teacher_ratio      453.0            0.096            0.132   

                       zeroshare_missing  zeroshare_present  \
frl_pct                            0.417              0.311   
teachers_fte                       0.584              0.322   
student_teacher_ratio              0.581              0.322   

                       median_enrl_missing  median_enrl_present  
frl_pct                              496.0                544.0  
teachers_fte                         259.0                541.0  
student_teacher_ratio                267.0                541.0  


In [14]:
# Association with location: free/reduced-lunch missingness rate by state (top/bottom 5)
by_state = (X_other.assign(frl_missing=X_other["frl_pct"].isna())
            .groupby("state_location")["frl_missing"].mean().sort_values())
print("lowest frl_pct missingness by state:\n", by_state.head(5).round(3))
print("\nhighest frl_pct missingness by state:\n", by_state.tail(5).round(3))

lowest frl_pct missingness by state:
 state_location
MS    0.0
NM    0.0
MI    0.0
ME    0.0
MD    0.0
Name: frl_missing, dtype: float64

highest frl_pct missingness by state:
 state_location
DE    1.0
DC    1.0
MT    1.0
MA    1.0
AK    1.0
Name: frl_missing, dtype: float64


The tables compare AP participation for schools with and without reported lunch and staffing data. Differences describe associations and may reflect state reporting practices.

## Missing-value strategies

In notebook 05 we use the same log transformations, scaling, one-hot encoding, and grade ordering for all three strategies. We compare XGBoost with missing values, random forest iterative imputation, and models trained on the available features for each missingness pattern.

## Verify and save

We save what the modeling notebook needs — and deliberately not a transformed/imputed development
matrix (that would bake in preprocessing leakage). Saved:

- raw development & test feature matrices, targets, and district groupings,
- feature-type definitions and the missing-value summary,
- the split indices + random seed (so the split is reproducible),
- a pointer to the reusable, *unfitted* functions in `preprocessing_utils.py`.

Reusable functions live in the module (imported by both notebooks) rather than being pickled.

In [ ]:
artifacts = {
    "feature_types": {"cont_skewed": cont_skewed, "cont_other": cont_other,
                      "cat_ftrs": cat_ftrs, "ord_ftrs": ord_ftrs,
                      "continuous_ftrs": continuous_ftrs},
    # RAW splits (NOT transformed / imputed) -> preprocessing is fit per fold in notebook 05
    "X_other": X_other, "X_test": X_test,
    "y_other": y_other, "y_test": y_test,
    "groups_other": groups_other, "groups_test": groups_test,
    "missing_summary": missing_summary,
    "split": {"other_idx": other_idx, "test_idx": test_idx,
              "random_state": RANDOM_STATE, "test_size": TEST_SIZE},
    "preprocessing_module": "preprocessing_utils",  # build_preprocessor / build_imputation_preprocessor / ReducedFeaturesRegressor
}
joblib.dump(artifacts, PROC_DIR / "04_preprocessing.joblib")

sz = (PROC_DIR / "04_preprocessing.joblib").stat().st_size / 1e6
print(f"saved {PROC_DIR/'04_preprocessing.joblib'}  ({sz:.1f} MB)")

saved ../data/processed/04_preprocessing.joblib  (3.1 MB)
NOTE: no transformed/imputed development matrix is saved (prevents CV leakage).
